[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week07.ipynb)

# Week 7: Regularization & Generalization
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

Overfitting; dropout, weight decay, early stopping; basic data augmentation.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Diagnose overfitting from the train and validation gap.
- Apply dropout, weight decay, early stopping, and augmentation.
- Attribute a generalization gain to a specific cause.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Force a model to overfit and show the train-minus-validation gap.

In [ ]:
# Force overfitting: tiny train set, large model
torch.manual_seed(0)
Xtr = torch.randn(30, 20); ytr = torch.randint(0, 2, (30,))
Xte = torch.randn(200, 20); yte = torch.randint(0, 2, (200,))
def fit(model, epochs=200, wd=0.0):
    o = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=wd); f = nn.CrossEntropyLoss()
    tr, te = [], []
    for _ in range(epochs):
        o.zero_grad(); f(model(Xtr), ytr).backward(); o.step()
        tr.append((model(Xtr).argmax(1) == ytr).float().mean().item())
        te.append((model(Xte).argmax(1) == yte).float().mean().item())
    return tr, te
big = lambda: nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))
tr, te = fit(big())
print(f"overfit: train {tr[-1]:.2f} vs test {te[-1]:.2f}")
plt.plot(tr, label="train"); plt.plot(te, label="test"); plt.legend(); plt.show()

## Demonstration 2
Add dropout and weight decay live and watch the gap close.

In [ ]:
# Dropout + weight decay shrink the train/test gap
reg = nn.Sequential(nn.Linear(20, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 2))
tr, te = fit(reg, wd=1e-2)
print(f"regularized: train {tr[-1]:.2f} vs test {te[-1]:.2f}")
plt.plot(tr, label="train"); plt.plot(te, label="test"); plt.legend(); plt.show()

## Demonstration 3
Demonstrate data augmentation on a few images.

In [ ]:
# Data augmentation (applied to TRAIN only)
from torchvision import transforms
img = torch.rand(3, 32, 32)
aug = transforms.Compose([transforms.RandomHorizontalFlip(p=1.0),
                          transforms.RandomCrop(32, padding=4)])
print("original", tuple(img.shape), "-> augmented", tuple(aug(img).shape))

---
Student materials for this week: the lab handout (`labs/week07.html`) and the curated references (`references/week07.html`) in the course site.